# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading, exploring, and analyzing the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and follows the [MLCommons Croissant format](https://mlcommons.org/croissant/).

In [ ]:
# Ensure `mlcroissant` package is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')

# Define the dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata as an object
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview

Review available record sets, fields, and their IDs. Croissant datasets structure data in *record sets* (tables or files), with each record set including *fields* (columns/attributes), each uniquely identified by their `@id`.

Let's enumerate all available record sets and fields using the metadata.

In [ ]:
# List all record sets with their IDs (using @id, as required)
record_sets = []
if hasattr(dataset.metadata, 'recordSets'):
    # mlcroissant >=0.10 stores 'recordSets' list
    record_sets = dataset.metadata.recordSets
elif hasattr(dataset.metadata, 'recordSet'):
    # Older versions or as per the provided dictionary
    record_sets = dataset.metadata.recordSet

print(f"Found {len(record_sets)} record sets:")
for rs in record_sets:
    print(f"  Record set name: {rs.name}")
    print(f"    @id: {rs.id}")
    if hasattr(rs, 'fields') and rs.fields:
        print("    Fields (@id):")
        for field in rs.fields:
            print(f"      - {field.name} (@id: {field.id})")
    print()

#### If you don't see any output above: 
Some Croissant schemas may not expose record sets in `dataset.metadata`. In those cases, you can programmatically inspect available record sets as follows:

In [ ]:
if not record_sets:
    # Try inspecting metadata as dict
    md = dataset.metadata.to_json()
    if 'recordSet' in md:
        print(f"recordSet found: {len(md['recordSet'])} available.")
        record_sets = md['recordSet']
        for rs in record_sets:
            print(f"  Record set name: {rs.get('name', '')}")
            print(f"    @id: {rs.get('@id')}")
            if 'field' in rs:
                for f in rs['field']:
                    print(f"      - {f.get('name', '(unknown)')} (@id: {f.get('@id')})")
            print()

# If still empty, fail gracefully
if not record_sets:
    print("No record sets found in this Croissant schema.")

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Reference record set and field `@id`s 

In [ ]:
# For this dataset, there may only be one main record set.
# Let's extract all available record sets into DataFrames (by @id).

from collections import OrderedDict

# Prefer list of record set @id's
record_set_ids = []
record_set_names = {}
record_set_fields = {}

# Use previous probing for main list
md_json = dataset.metadata.to_json()
rs_list = md_json.get('recordSet', [])
for rs in rs_list:
    rsid = rs.get('@id')
    name = rs.get('name', '')
    record_set_ids.append(rsid)
    record_set_names[rsid] = name
    record_set_fields[rsid] = rs.get('field', [])

dataframes = dict()
for record_set_id in record_set_ids:
    print(f"\nLoading records from: {record_set_id} ({record_set_names[record_set_id]})")
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"  Loaded DataFrame with shape: {df.shape}")
            print(f"  Columns: {df.columns.tolist()}")
            print(df.head(2))
        else:
            print("  (No records available or failed to load.")
    except Exception as e:
        print(f"  Error loading records: {e}")

# Choose the first available record set for further analysis
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    df_main = dataframes[main_record_set_id]
    print(f"\nColumns in Main DataFrame ({main_record_set_id}):\n", df_main.columns.tolist())
    display(df_main.head())
else:
    print('No record set IDs available to extract.')

## 4. Exploratory Data Analysis (EDA)

Let's apply common data processing steps such as filtering protein records by a numeric field, normalizing protein molecular weights or abundances, removing outliers, or grouping by a categorical field.

In [ ]:
import numpy as np

# Identify a numeric field for demonstration, e.g., peptide count or molecular weight
# Field IDs as per Croissant schema: e.g., 'peptide_count' or 'molecular_weight'
possible_numeric_fields = [col for col in df_main.columns if ('peptide' in col.lower() or 'count' in col.lower() or 'weight' in col.lower() or 'abundance' in col.lower())]
print(f"Possible numeric fields: {possible_numeric_fields}")

# Choose one for demo (preferably 'molecular_weight')
if 'molecular_weight' in df_main.columns:
    numeric_field = 'molecular_weight'
elif possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]
else:
    raise ValueError('No obvious numeric field found for EDA.')

# Filter for numeric values (remove NaNs or strings)
df_main_filtered = df_main[pd.to_numeric(df_main[numeric_field], errors='coerce').notna()]
df_main_filtered[numeric_field] = pd.to_numeric(df_main_filtered[numeric_field], errors='coerce')

# Remove outliers (e.g., beyond 1st and 99th percentile)
lo, hi = df_main_filtered[numeric_field].quantile([0.01, 0.99])
filtered_df = df_main_filtered[(df_main_filtered[numeric_field] > lo) & (df_main_filtered[numeric_field] < hi)]
print(f"Filtered {numeric_field} between percentiles: [{lo:.2f}, {hi:.2f}]; retained {filtered_df.shape[0]} rows.")

# Normalize field (z-score)
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Grouping example: by protein type/description, if available
possible_group_fields = [col for col in df_main.columns if ('description' in col.lower() or 'type' in col.lower() or 'category' in col.lower())]
group_field = possible_group_fields[0] if possible_group_fields else None

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].agg(['mean', 'count'])
    print(f"\nGrouped data by {group_field}:")
    print(grouped_df.head())
else:
    print("No suitable categorical group field found for grouping analysis.")

## 5. Visualization

Visualize the numeric field distribution (histogram) and, if possible, the mean value per group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
plt.figure(figsize=(8,5))
sns.histplot(filtered_df[numeric_field], bins=40, kde=True, color='dodgerblue')
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Frequency")
plt.show()

# If grouping field exists, plot top N by mean
if group_field:
    top_groups = grouped_df.sort_values('mean', ascending=False).head(10)
    plt.figure(figsize=(10,5))
    sns.barplot(x=top_groups.index, y=top_groups['mean'], palette="crest")
    plt.title(f"Top 10 {group_field} by mean {numeric_field}")
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {numeric_field}")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion

We demonstrated loading and initial exploration of the FAIR^2 mass spectrometry dataset using the `mlcroissant` library. With programmatic access to record set structures and fields via `@id`, it's possible to filter, normalize, and visualize protein abundance and molecular property data in a reproducible and FAIR way.

- This dataset contains protein-level data from human mast cell extracellular vesicles, with fields like molecular weight, peptide count, and annotations.
- Croissant schema ensures all data objects (record sets, fields) are referenced by their `@id` for reliable FAIR access.
- Further steps could include advanced modeling or integration with domain bioinformatics tools.

For more, see the [mlcroissant documentation](https://github.com/mlcommons/croissant/tree/main/python/mlcroissant).